# Second Model: Random Forest and XGBoost vs. the Logistic Regression Baseline

Compares Random Forest and XGBoost against the logistic regression baseline (`log-reg-model.ipynb`) using the **same 2024 train / 2025 test split** (`data/X_train.csv`, etc.) and the **same metrics** (accuracy, precision, recall, F1, ROC-AUC). The preprocessing is copied from the baseline so any difference comes from the model, not the pipeline.

Sections:
1. Same-split comparison with default-ish settings
2. Light tuning (grid search using training data only, then one look at the test set)
3. Does the model just learn which school is hosting? (drop team names, team-only reference)
4. Performance by conference group (fairness check)
5. Summary

In [3]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()
print(X_train.shape, X_test.shape)

(815, 37) (858, 37)


## 1. Same split, same preprocessing, same metrics

The preprocessing matches the baseline: median-impute and scale numeric columns, most-frequent-impute and one-hot encode text columns. Trees do not need scaling, but keeping one pipeline for all three models keeps the comparison fair.

Random Forest uses 500 trees. XGBoost uses 300 shallow trees (depth 4, learning rate 0.05). Neither is tuned yet.

In [4]:
def make_preprocessor(X):
    cat = X.select_dtypes(include=["object", "str", "bool", "category"]).columns.tolist()
    num = [c for c in X.columns if c not in cat]
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
    ])

def metrics(name, y_true, y_pred, y_proba):
    return {"model": name,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred),
            "recall": recall_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred),
            "roc_auc": roc_auc_score(y_true, y_proba)}

def fit_eval(name, clf, drop=()):
    a, b = X_train.drop(columns=list(drop)), X_test.drop(columns=list(drop))
    pipe = Pipeline([("preprocessor", make_preprocessor(a)), ("classifier", clf)]).fit(a, y_train)
    proba = pipe.predict_proba(b)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return metrics(name, y_test, pred, proba), pipe, pred, proba

default_models = {
    "Logistic regression (baseline)": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, random_state=42, eval_metric="logloss"),
}
rows = [fit_eval(n, m)[0] for n, m in default_models.items()]
results_default = pd.DataFrame(rows).set_index("model").round(3)
results_default

,accuracy,precision,recall,f1,roc_auc
model,,,,,
Logistic regression (baseline),0.815,0.786,0.792,0.789,0.893
Random Forest,0.794,0.784,0.728,0.755,0.886
XGBoost,0.797,0.783,0.741,0.762,0.882


**Reading this:** the logistic regression baseline reproduces the numbers in `log-reg-model.ipynb` (F1 0.789, ROC-AUC 0.893). Out of the box, Random Forest and XGBoost do **not** beat it.

## 2. Light tuning

Grid search with 5-fold cross-validation on the **2024 training data only**, scored on F1. The 2025 test set is only used once at the end, so tuning does not leak.

Note: cross-validation inside 2024 is optimistic (random folds mix games from the same weeks), so the CV F1 is higher than the test F1. That is expected, and it is a reason to keep the time-based split for the real score.

In [5]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
grids = {
    "Random Forest": (RandomForestClassifier(random_state=42, n_jobs=-1),
        {"classifier__n_estimators": [300, 600], "classifier__max_depth": [6, 10, None],
         "classifier__min_samples_leaf": [1, 3, 5], "classifier__max_features": ["sqrt", 0.5]}),
    "XGBoost": (XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=4),
        {"classifier__n_estimators": [200, 400], "classifier__max_depth": [2, 3, 4],
         "classifier__learning_rate": [0.03, 0.07], "classifier__subsample": [0.8],
         "classifier__colsample_bytree": [0.6, 0.9], "classifier__min_child_weight": [1, 5]}),
}

def tune(name, drop=()):
    clf, grid = grids[name]
    a, b = X_train.drop(columns=list(drop)), X_test.drop(columns=list(drop))
    pipe = Pipeline([("preprocessor", make_preprocessor(a)), ("classifier", clf)])
    gs = GridSearchCV(pipe, grid, scoring="f1", cv=cv, n_jobs=1).fit(a, y_train)
    best = gs.best_estimator_
    proba = best.predict_proba(b)[:, 1]
    r = metrics(name + " (tuned)", y_test, (proba >= 0.5).astype(int), proba)
    r["cv_f1_train"] = gs.best_score_
    return r, best, gs.best_params_

tuned_rows, tuned_models = [], {}
for n in ["Random Forest", "XGBoost"]:
    r, best, params = tune(n)
    tuned_rows.append(r); tuned_models[n] = best
    print(n, params)
pd.concat([results_default.reset_index(), pd.DataFrame(tuned_rows)]).set_index("model").round(3)

Random Forest {'classifier__max_depth': None, 'classifier__max_features': 0.5, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 600}
XGBoost {'classifier__colsample_bytree': 0.6, 'classifier__learning_rate': 0.07, 'classifier__max_depth': 3, 'classifier__min_child_weight': 1, 'classifier__n_estimators': 400, 'classifier__subsample': 0.8}


,accuracy,precision,recall,f1,roc_auc,cv_f1_train
model,,,,,,
Logistic regression (baseline),0.815,0.786,0.792,0.789,0.893,NaN
Random Forest,0.794,0.784,0.728,0.755,0.886,NaN
XGBoost,0.797,0.783,0.741,0.762,0.882,NaN
Random Forest (tuned),0.818,0.812,0.760,0.785,0.893,0.845
XGBoost (tuned),0.787,0.773,0.725,0.748,0.881,0.820


**Reading this:** tuned Random Forest lands at the baseline (F1 about 0.79, ROC-AUC 0.893). Tuned XGBoost stays a bit below. Neither model clears the 0.80 F1 target, and the differences between them are within noise for 858 test games.

## 3. Does the model just learn which school is hosting?

The baseline uses `homeTeam` and `awayTeam` as one-hot columns, which lets the model memorize each school's usual attendance. Two checks:

1. Drop `homeTeam` and `awayTeam` and re-run all three models.
2. A **team-only reference** with no model at all: predict each home team's 2025 outcome from its 2024 high-attendance rate.

In [6]:
no_team = ["homeTeam", "awayTeam"]
rows = []
for n, m in default_models.items():
    rows.append(fit_eval(n + " (no team names)", m, drop=no_team)[0])
for n in ["Random Forest", "XGBoost"]:
    r, _, _ = tune(n, drop=no_team)
    rows.append({**r, "model": n + " (tuned, no team names)"})
pd.DataFrame(rows).set_index("model").round(3)

,accuracy,precision,recall,f1,roc_auc,cv_f1_train
model,,,,,,
Logistic regression (baseline) (no team names),0.768,0.735,0.733,0.734,0.838,NaN
Random Forest (no team names),0.788,0.780,0.717,0.747,0.870,NaN
XGBoost (no team names),0.784,0.768,0.725,0.746,0.873,NaN
"Random Forest (tuned, no team names)",0.788,0.781,0.715,0.747,0.870,0.799
"XGBoost (tuned, no team names)",0.788,0.770,0.733,0.751,0.874,0.818


In [7]:
# Team-only reference: each home team's 2024 high-attendance rate predicts its 2025 games
team_rate = pd.DataFrame({"team": X_train["homeTeam"], "y": y_train}).groupby("team")["y"].mean()
score = X_test["homeTeam"].map(team_rate).fillna(y_train.mean())   # unseen teams get the overall rate
pred = (score >= 0.5).astype(int)
print("Team-only reference (no model):")
print(pd.Series({k: v for k, v in metrics("team-only", y_test, pred, score).items() if k != "model"}).round(3).to_string())
print("test games whose home team was not in 2024 training:", X_test["homeTeam"].map(team_rate).isna().sum())

Team-only reference (no model):
accuracy     0.830
precision    0.788
recall       0.835
f1           0.811
roc_auc      0.882
test games whose home team was not in 2024 training: 29


**Reading this:**
- Without team names, logistic regression drops (F1 0.789 to 0.734, ROC-AUC 0.893 to 0.838), while Random Forest and XGBoost drop much less (F1 about 0.75, ROC-AUC about 0.87). So the second model is better at using the game-level features, and logistic regression leans more on school identity.
- The team-only reference, which uses no features at all, scores F1 0.81 and ROC-AUC 0.88. Most of the baseline's score comes from knowing which school is hosting, not from game-specific drivers. That is the main thing to discuss with Christina.

## 4. Performance by conference group (fairness check)

Group 1 is the Power 4 (SEC, Big Ten, Big 12, ACC). Group 2 is every other FBS home team. Same 2025 test set, same 0.5 threshold.

In [8]:
P4 = ["SEC", "Big Ten", "Big 12", "ACC"]
group = np.where(X_test["homeConference"].isin(P4), "Power 4", "Other FBS")
print("actual high-attendance rate:", y_test.groupby(group).mean().round(3).to_dict())

fair_rows = []
for n, m in default_models.items():
    _, _, pred, proba = fit_eval(n, m)
    for g in ["Power 4", "Other FBS"]:
        k = group == g
        tn, fp, fn, tp = confusion_matrix(y_test[k], pred[k]).ravel()
        fair_rows.append({"model": n, "group": g, "games": int(k.sum()),
                          "actual_high": y_test[k].mean(), "predicted_high": pred[k].mean(),
                          "recall (high games caught)": tp / (tp + fn),
                          "false_alarm_rate": fp / (fp + tn),
                          "f1": f1_score(y_test[k], pred[k])})
pd.DataFrame(fair_rows).set_index(["model", "group"]).round(3)

actual high-attendance rate: {'Other FBS': 0.194, 'Power 4': 0.661}


games  actual_high  predicted_high  \
model                          group                                           
Logistic regression (baseline) Power 4      446        0.661           0.747   
                               Other FBS    412        0.194           0.109   
Random Forest                  Power 4      446        0.661           0.735   
                               Other FBS    412        0.194           0.049   
XGBoost                        Power 4      446        0.661           0.713   
                               Other FBS    412        0.194           0.090   

                                          recall (high games caught)  \
model                          group                                   
Logistic regression (baseline) Power 4                         0.892   
                               Other FBS                       0.425   
Random Forest                  Power 4                         0.864   
                               Other FBS                       0.225   
XGBoost                        Power 4                         0.844   
                               Other FBS                       0.362   

                                          false_alarm_rate     f1  
model                          group                               
Logistic regression (baseline) Power 4               0.464  0.838  
                               Other FBS             0.033  0.544  
Random Forest                  Power 4               0.483  0.819  
                               Other FBS             0.006  0.360  
XGBoost                        Power 4               0.457  0.812  
                               Other FBS             0.024  0.496

**Reading this:** all three models catch most high-attendance games at Power 4 schools (recall 84 to 89%) but miss most of them at other schools (recall only 23 to 43%). They also over-predict "high" for Power 4 games that end up low (false alarm rate about 45%). The F1 gap comes from the data pattern: high attendance is 66% of Power 4 games and 19% of other games.

## 5. Summary

| Model (2024 train, 2025 test) | F1 | ROC-AUC |
|---|---|---|
| Logistic regression (baseline) | 0.789 | 0.893 |
| Random Forest (default) | 0.755 | 0.886 |
| XGBoost (default) | 0.762 | 0.885 |
| Random Forest (tuned) | 0.786 | 0.893 |
| XGBoost (tuned) | 0.761 | 0.882 |
| Team-only reference (no model) | 0.811 | 0.882 |

Takeaways: the second model ties the baseline, no model reaches a clear gain, and a lookup of each school's past attendance rate does as well as any of them. Getting past 0.8 in a meaningful way will take features that vary game to game within a school, not a more complex model.